# AERONET Diagnostics: Is the Addis Anomaly Surface-Level or Columnar?

**Core question**: Is the anomalous HIPS behavior at Addis a surface-level measurement artifact, or does it reflect something genuinely unusual about aerosol vertical distribution?

## Approach
We have three measurement types:
- **AERONET** = columnar (entire atmosphere) optical properties
- **HIPS/FTIR** = surface-level filter-based BC mass concentration
- **Aethalometer** = surface-level light absorption → BC

The bridge is **light absorption** — the aethalometer and AERONET are both optical instruments.

## Experiments
1. **Sub-daily BC vs AOD matching** — diurnal decoupling analysis (highest value)
2. **Boundary layer trapping signal** — BC spikes without AOD response
3. **Coarse mode AOD as dust proxy** — stronger than XRF iron for dust interference
4. **Surface AAE vs columnar AE** — do sources look different from surface vs column?
5. **Fine Mode Fraction × BC regimes** — when FMF is high but BC is low (or vice versa), what's going on?

---

## Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Add research scripts to path
RESEARCH_DIR = '/Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem'
scripts_path = os.path.join(RESEARCH_DIR, 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, FILTER_DATA_PATH, PROCESSED_SITES_DIR

# Addis Ababa season definitions
SEASONS = {
    'Dry': [10, 11, 12, 1, 2],
    'Belg': [3, 4, 5],
    'Kiremt': [6, 7, 8, 9]
}
SEASON_COLORS = {'Dry': '#E67E22', 'Belg': '#27AE60', 'Kiremt': '#3498DB'}
SEASON_ORDER = ['Dry', 'Belg', 'Kiremt']

def assign_season(month):
    for name, months in SEASONS.items():
        if month in months:
            return name
    return 'Unknown'

AERONET_MISSING = -999.

print('Setup complete')

## Data Loading

We load:
1. **Minute-level BC** — for sub-daily matching with AERONET all-points
2. **AERONET all-points AOD** (~24k observations with exact timestamps)
3. **AERONET all-points SDA** (fine/coarse mode decomposition)
4. **HIPS/FTIR filter data** (for connecting surface-column)

In [ ]:
# === Paths ===
BC_MINUTE_PATH = (
    '/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/'
    'My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/'
    'Aethelometry Data/JacrosMA350 60s Data20250804082112/'
    'df_jacros_cleaned_API_and_OG_manual_BC_all_wl.pkl'
)

AERONET_BASE = (
    '/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/'
    'My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/'
    'AERONET/Jacros'
)
AERONET_AOD_ALLPTS = os.path.join(
    AERONET_BASE,
    '20220101_20251231_AAU_Jackros_ET all',
    '20220101_20251231_AAU_Jackros_ET.lev20'
)
AERONET_SDA_ALLPTS = os.path.join(
    AERONET_BASE,
    '20220101_20251231_AAU_Jackros_ET all',
    '20220101_20251231_AAU_Jackros_ET.ONEILL_lev20'
)
AERONET_AOD_DAILY = os.path.join(
    AERONET_BASE,
    '20220101_20251231_AAU_Jackros_ET Daily',
    '20220101_20251231_AAU_Jackros_ET.lev20'
)
AERONET_SDA_DAILY = os.path.join(
    AERONET_BASE,
    '20220101_20251231_AAU_Jackros_ET Daily',
    '20220101_20251231_AAU_Jackros_ET.ONEILL_lev20'
)

print('Paths configured')

In [ ]:
# === Load minute-level BC ===
bc_raw = pd.read_pickle(BC_MINUTE_PATH)
bc_raw['datetime_local'] = pd.to_datetime(bc_raw['datetime_local'])
bc_raw.set_index('datetime_local', inplace=True)
bc_raw.sort_index(inplace=True)

# Convert ng/m³ → µg/m³
for col in ['UV BCc', 'IR BCc', 'UV BC1', 'IR BC1']:
    if col in bc_raw.columns:
        bc_raw[col] = bc_raw[col] / 1000

# Basic QC: negative and 3-sigma outliers
for col in ['UV BCc', 'IR BCc']:
    if col in bc_raw.columns:
        bc_raw.loc[bc_raw[col] < 0, col] = np.nan
        mu, sigma = bc_raw[col].mean(), bc_raw[col].std()
        bc_raw.loc[bc_raw[col] > mu + 3 * sigma, col] = np.nan

print(f'BC minute data: {len(bc_raw):,} records')
print(f'  Range: {bc_raw.index.min()} → {bc_raw.index.max()}')
print(f'  IR BCc valid: {bc_raw["IR BCc"].notna().sum():,}')

In [ ]:
# === Load AERONET all-points AOD ===
def load_aeronet_allpoints_aod(filepath):
    """Load AERONET all-points AOD with UTC timestamps."""
    df = pd.read_csv(filepath, skiprows=6)
    # Parse datetime: date is dd:mm:yyyy, time is hh:mm:ss (UTC)
    df['datetime_utc'] = pd.to_datetime(
        df['Date(dd:mm:yyyy)'] + ' ' + df['Time(hh:mm:ss)'],
        format='%d:%m:%Y %H:%M:%S'
    )
    df.set_index('datetime_utc', inplace=True)
    df.sort_index(inplace=True)
    df.replace(AERONET_MISSING, np.nan, inplace=True)
    return df

aod_all = load_aeronet_allpoints_aod(AERONET_AOD_ALLPTS)
print(f'AERONET AOD all-points: {len(aod_all):,} observations')
print(f'  Range: {aod_all.index.min()} → {aod_all.index.max()}')
print(f'  AOD_500nm valid: {aod_all["AOD_500nm"].notna().sum():,}')

In [ ]:
# === Load AERONET all-points SDA ===
def load_aeronet_allpoints_sda(filepath):
    """Load AERONET SDA all-points with UTC timestamps."""
    df = pd.read_csv(filepath, skiprows=6)
    df['datetime_utc'] = pd.to_datetime(
        df['Date_(dd:mm:yyyy)'] + ' ' + df['Time_(hh:mm:ss)'],
        format='%d:%m:%Y %H:%M:%S'
    )
    df.set_index('datetime_utc', inplace=True)
    df.sort_index(inplace=True)
    df.replace(AERONET_MISSING, np.nan, inplace=True)
    return df

sda_all = load_aeronet_allpoints_sda(AERONET_SDA_ALLPTS)
print(f'AERONET SDA all-points: {len(sda_all):,} observations')
print(f'  Fine Mode AOD valid: {sda_all["Fine_Mode_AOD_500nm[tau_f]"].notna().sum():,}')
print(f'  Coarse Mode AOD valid: {sda_all["Coarse_Mode_AOD_500nm[tau_c]"].notna().sum():,}')

In [ ]:
# === Load daily AERONET for reference ===
def load_aeronet_daily(filepath, date_col='Date(dd:mm:yyyy)'):
    df = pd.read_csv(filepath, skiprows=6)
    df['Date'] = pd.to_datetime(df[date_col], format='%d:%m:%Y')
    df.set_index('Date', inplace=True)
    df.sort_index(inplace=True)
    df.replace(AERONET_MISSING, np.nan, inplace=True)
    return df

aod_daily = load_aeronet_daily(AERONET_AOD_DAILY)
sda_daily = load_aeronet_daily(AERONET_SDA_DAILY, date_col='Date_(dd:mm:yyyy)')
print(f'AERONET daily AOD: {len(aod_daily)} days')
print(f'AERONET daily SDA: {len(sda_daily)} days')

In [ ]:
# === Load HIPS/FTIR filter data ===
# Data is in long format (one row per parameter per filter) — pivot to wide
filters_long = pd.read_pickle(FILTER_DATA_PATH)
addis_long = filters_long[filters_long['Site'] == 'ETAD'].copy()
print(f'Addis filter rows (long): {len(addis_long)}')
print(f'Unique FilterIds: {addis_long["FilterId"].nunique()}')

# Pivot: one row per FilterId with parameters as columns
addis_filters = addis_long.pivot_table(
    index=['FilterId', 'SampleDate'],
    columns='Parameter',
    values='Concentration',
    aggfunc='first'
).reset_index()

# Parse dates
addis_filters['filter_date'] = pd.to_datetime(addis_filters['SampleDate']).dt.normalize()

print(f'Addis filter samples (wide): {len(addis_filters)}')
print(f'Date range: {addis_filters["filter_date"].min()} → {addis_filters["filter_date"].max()}')

# Check key columns
for col in ['HIPS_Fabs', 'EC_ftir', 'ChemSpec_Iron_PM2.5']:
    n = addis_filters[col].notna().sum() if col in addis_filters.columns else 0
    print(f'  {col}: {n} valid')

---

## Experiment 1: Sub-daily BC vs AOD Temporal Matching

**Goal**: Match individual AERONET observations (daytime only, ~every 15 min) with concurrent aethalometer BC.

AERONET timestamps are UTC. Addis Ababa is UTC+3. We'll:
1. Convert BC timestamps to UTC
2. For each AERONET observation, average BC within ±15 min
3. Look at the correlation structure by time of day and season

In [ ]:
# Convert BC to UTC for matching
# BC index is local time (UTC+3), strip any tz info and subtract 3h
bc_utc = bc_raw[['IR BCc', 'UV BCc']].copy()
if bc_utc.index.tz is not None:
    bc_utc.index = bc_utc.index.tz_convert('UTC').tz_localize(None)
else:
    # Assume local time is UTC+3
    bc_utc.index = bc_utc.index - pd.Timedelta(hours=3)

print(f'BC (UTC): {bc_utc.index.min()} → {bc_utc.index.max()}')
print(f'AERONET:  {aod_all.index.min()} → {aod_all.index.max()}')

In [ ]:
# Match AERONET observations with ±15 min BC average
MATCH_WINDOW = pd.Timedelta(minutes=15)

matched_rows = []
aod_cols = ['AOD_500nm', 'AOD_440nm', 'AOD_870nm', 'Precipitable_Water(cm)',
            '440-870_Angstrom_Exponent', 'Solar_Zenith_Angle(Degrees)']
sda_cols = ['Total_AOD_500nm[tau_a]', 'Fine_Mode_AOD_500nm[tau_f]',
            'Coarse_Mode_AOD_500nm[tau_c]', 'FineModeFraction_500nm[eta]',
            'Angstrom_Exponent(AE)-Total_500nm[alpha]']

# Merge AOD and SDA on their shared timestamps
aeronet_merged = aod_all[aod_cols].join(sda_all[sda_cols], how='outer')

for t_aero, row in aeronet_merged.iterrows():
    # Get BC within window
    mask = (bc_utc.index >= t_aero - MATCH_WINDOW) & (bc_utc.index <= t_aero + MATCH_WINDOW)
    bc_window = bc_utc.loc[mask]
    
    if len(bc_window) < 5:  # need at least 5 minutes of data
        continue
    
    ir_bc = bc_window['IR BCc'].mean()
    uv_bc = bc_window['UV BCc'].mean()
    n_bc = bc_window['IR BCc'].notna().sum()
    
    if pd.isna(ir_bc):
        continue
    
    rec = {
        'datetime_utc': t_aero,
        'datetime_local': t_aero + pd.Timedelta(hours=3),
        'IR_BCc': ir_bc,
        'UV_BCc': uv_bc,
        'n_bc_minutes': n_bc,
    }
    for col in aod_cols + sda_cols:
        rec[col] = row.get(col, np.nan)
    
    matched_rows.append(rec)

matched = pd.DataFrame(matched_rows)
matched['hour_local'] = matched['datetime_local'].dt.hour
matched['month'] = matched['datetime_local'].dt.month
matched['season'] = matched['month'].map(assign_season)
matched['date'] = matched['datetime_local'].dt.date

print(f'Matched observations: {len(matched):,}')
print(f'  Date range: {matched["datetime_local"].min()} → {matched["datetime_local"].max()}')
print(f'  Unique days: {matched["date"].nunique()}')
print(f'\nSeason breakdown:')
print(matched['season'].value_counts())

In [ ]:
# Quick look at the matched data
print('Matched data summary:')
key_cols = ['IR_BCc', 'AOD_500nm', 'Fine_Mode_AOD_500nm[tau_f]',
            'Coarse_Mode_AOD_500nm[tau_c]', 'FineModeFraction_500nm[eta]',
            'Precipitable_Water(cm)', '440-870_Angstrom_Exponent']
available = [c for c in key_cols if c in matched.columns]
print(matched[available].describe().round(4))

---

## Experiment 2: Diurnal Decoupling — Boundary Layer Trapping Signal

**Key hypothesis**: If BC spikes in the morning (shallow boundary layer, emissions trapped) but columnar AOD doesn't respond proportionally, that indicates the aerosol loading is confined to the surface layer.

At 2370m elevation, Addis has a unique boundary layer dynamic. We compare the diurnal cycle of surface BC vs columnar AOD.

In [ ]:
# Diurnal profiles: BC vs AOD by hour of day
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, season in enumerate(SEASON_ORDER):
    ax = axes[idx]
    sdata = matched[matched['season'] == season].copy()
    
    if len(sdata) < 10:
        ax.set_title(f'{season} (n={len(sdata)}, insufficient data)')
        continue
    
    # Hourly means
    hourly_bc = sdata.groupby('hour_local')['IR_BCc'].agg(['mean', 'std', 'count'])
    hourly_aod = sdata.groupby('hour_local')['AOD_500nm'].agg(['mean', 'std', 'count'])
    
    ax2 = ax.twinx()
    
    # BC
    ax.errorbar(hourly_bc.index, hourly_bc['mean'],
                yerr=hourly_bc['std'] / np.sqrt(hourly_bc['count']),
                fmt='o-', color='#E74C3C', linewidth=2, markersize=6,
                label='Surface BC', capsize=3)
    
    # AOD
    valid_aod = hourly_aod.dropna(subset=['mean'])
    ax2.errorbar(valid_aod.index, valid_aod['mean'],
                 yerr=valid_aod['std'] / np.sqrt(valid_aod['count']),
                 fmt='s-', color='#3498DB', linewidth=2, markersize=6,
                 label='AOD 500nm', capsize=3)
    
    ax.set_xlabel('Hour (local)', fontsize=11)
    ax.set_ylabel('BC (µg/m³)', fontsize=11, color='#E74C3C')
    ax2.set_ylabel('AOD 500nm', fontsize=11, color='#3498DB')
    ax.set_title(f'{season} (n={len(sdata)})', fontsize=12, fontweight='bold')
    
    # Combined legend
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Diurnal Decoupling: Surface BC vs Columnar AOD', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compute BC/AOD ratio as a proxy for "how much of the column is surface"
valid_match = matched.dropna(subset=['IR_BCc', 'AOD_500nm']).copy()
valid_match = valid_match[valid_match['AOD_500nm'] > 0.01]  # avoid division by near-zero
valid_match['BC_AOD_ratio'] = valid_match['IR_BCc'] / valid_match['AOD_500nm']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BC/AOD ratio diurnal cycle
ax = axes[0]
for season in SEASON_ORDER:
    sdata = valid_match[valid_match['season'] == season]
    hourly = sdata.groupby('hour_local')['BC_AOD_ratio'].mean()
    ax.plot(hourly.index, hourly.values, 'o-', color=SEASON_COLORS[season],
            linewidth=2, markersize=6, label=season)

ax.set_xlabel('Hour (local)', fontsize=11)
ax.set_ylabel('BC / AOD_500nm ratio', fontsize=11)
ax.set_title('BC-to-AOD Ratio Diurnal Cycle\n(higher = more surface-confined)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# BC/AOD ratio by season boxplot
ax = axes[1]
plot_data = [valid_match[valid_match['season'] == s]['BC_AOD_ratio'].dropna() for s in SEASON_ORDER]
bp = ax.boxplot(plot_data, labels=SEASON_ORDER, patch_artist=True, showfliers=False)
for patch, season in zip(bp['boxes'], SEASON_ORDER):
    patch.set_facecolor(SEASON_COLORS[season])
    patch.set_alpha(0.7)
ax.set_ylabel('BC / AOD_500nm ratio', fontsize=11)
ax.set_title('BC/AOD Ratio by Season', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Stats
print('BC/AOD ratio by season:')
for season in SEASON_ORDER:
    s = valid_match[valid_match['season'] == season]['BC_AOD_ratio']
    print(f'  {season}: mean={s.mean():.2f}, median={s.median():.2f}, n={len(s)}')

In [ ]:
# Morning vs afternoon decoupling
# AERONET only measures during daytime; at Addis (near equator) that's ~6am-6pm local
# Morning: 6-10 local (shallow BL), Afternoon: 12-16 local (deep BL)
valid_match['period'] = 'midday'
valid_match.loc[valid_match['hour_local'].between(6, 9), 'period'] = 'morning'
valid_match.loc[valid_match['hour_local'].between(13, 16), 'period'] = 'afternoon'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, season in enumerate(SEASON_ORDER):
    ax = axes[idx]
    sdata = valid_match[valid_match['season'] == season]
    
    for period, marker, color in [('morning', 'o', '#E74C3C'), 
                                    ('midday', 's', '#9B59B6'),
                                    ('afternoon', '^', '#3498DB')]:
        pdata = sdata[sdata['period'] == period]
        if len(pdata) > 3:
            ax.scatter(pdata['IR_BCc'], pdata['AOD_500nm'], marker=marker,
                      alpha=0.4, s=30, label=f'{period} (n={len(pdata)})', color=color)
            # Regression line
            clean = pdata[['IR_BCc', 'AOD_500nm']].dropna()
            if len(clean) > 5:
                slope, intercept, r, p, _ = stats.linregress(clean['IR_BCc'], clean['AOD_500nm'])
                x_fit = np.linspace(clean['IR_BCc'].min(), clean['IR_BCc'].max(), 50)
                ax.plot(x_fit, slope * x_fit + intercept, '--', color=color, alpha=0.7, linewidth=1.5)
    
    ax.set_xlabel('BC (µg/m³)', fontsize=11)
    ax.set_ylabel('AOD 500nm' if idx == 0 else '', fontsize=11)
    ax.set_title(f'{season}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('BC vs AOD by Time of Day\n(different slopes = boundary layer effect)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print regression stats
print('\nBC vs AOD regression by period and season:')
for season in SEASON_ORDER:
    print(f'\n  {season}:')
    for period in ['morning', 'midday', 'afternoon']:
        pdata = valid_match[(valid_match['season'] == season) & (valid_match['period'] == period)]
        clean = pdata[['IR_BCc', 'AOD_500nm']].dropna()
        if len(clean) > 5:
            slope, intercept, r, p, _ = stats.linregress(clean['IR_BCc'], clean['AOD_500nm'])
            print(f'    {period:10s}: slope={slope:.4f}, r={r:.3f}, p={p:.2e}, n={len(clean)}')

---

## Experiment 3: Coarse Mode AOD as Dust Proxy

**Hypothesis**: The anomalous HIPS behavior at Addis may be related to dust interference. AERONET's coarse-mode AOD (from SDA) is a direct measure of dust loading in the column — much more robust than using XRF iron as an indirect surface proxy.

If days with high coarse-mode AOD correspond to anomalous HIPS-FTIR disagreement, that's strong evidence for dust interference.

In [ ]:
# Daily coarse mode AOD from SDA
coarse_col = 'Coarse_Mode_AOD_500nm[tau_c]'
fine_col = 'Fine_Mode_AOD_500nm[tau_f]'
fmf_col = 'FineModeFraction_500nm[eta]'

# Use daily SDA data
sda_daily['month'] = sda_daily.index.month
sda_daily['season'] = sda_daily['month'].map(assign_season)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Coarse vs Fine AOD
ax = axes[0]
valid_sda = sda_daily.dropna(subset=[coarse_col, fine_col])
for season in SEASON_ORDER:
    s = valid_sda[valid_sda['season'] == season]
    ax.scatter(s[fine_col], s[coarse_col], color=SEASON_COLORS[season],
              alpha=0.5, s=30, label=f'{season} (n={len(s)})', edgecolors='k', linewidth=0.3)
ax.set_xlabel('Fine Mode AOD 500nm', fontsize=11)
ax.set_ylabel('Coarse Mode AOD 500nm', fontsize=11)
ax.set_title('Fine vs Coarse Mode AOD', fontsize=12, fontweight='bold')
ax.plot([0, ax.get_xlim()[1]], [0, ax.get_xlim()[1]], 'k--', alpha=0.3, label='1:1')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Coarse mode AOD time series
ax = axes[1]
valid = sda_daily[coarse_col].dropna()
ax.plot(valid.index, valid.values, 'o', markersize=3, alpha=0.5, color='#8B4513')
rolling = valid.rolling(30, min_periods=7).mean()
ax.plot(rolling.index, rolling.values, '-', linewidth=2, color='#8B4513')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Coarse Mode AOD 500nm', fontsize=11)
ax.set_title('Coarse Mode AOD Time Series\n(dust proxy)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# Coarse mode by season
ax = axes[2]
plot_data = [valid_sda[valid_sda['season'] == s][coarse_col].dropna() for s in SEASON_ORDER]
bp = ax.boxplot(plot_data, labels=SEASON_ORDER, patch_artist=True, showfliers=True,
               flierprops=dict(marker='o', markersize=3, alpha=0.3))
for patch, season in zip(bp['boxes'], SEASON_ORDER):
    patch.set_facecolor(SEASON_COLORS[season])
    patch.set_alpha(0.7)
ax.set_ylabel('Coarse Mode AOD 500nm', fontsize=11)
ax.set_title('Coarse AOD by Season', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Coarse Mode AOD — Columnar Dust Loading', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Coarse Mode AOD by season:')
for season in SEASON_ORDER:
    s = valid_sda[valid_sda['season'] == season][coarse_col]
    print(f'  {season}: mean={s.mean():.4f}, median={s.median():.4f}, n={len(s)}')

In [ ]:
# Connect coarse mode AOD with HIPS/FTIR filter data
# filter_date was already parsed in cell a9

# Prepare HIPS and FTIR columns
hips_col = 'HIPS_Fabs' if 'HIPS_Fabs' in addis_filters.columns else None
ftir_col = 'EC_ftir' if 'EC_ftir' in addis_filters.columns else None
iron_col = 'ChemSpec_Iron_PM2.5' if 'ChemSpec_Iron_PM2.5' in addis_filters.columns else None

print(f'HIPS column: {hips_col} ({addis_filters[hips_col].notna().sum() if hips_col else 0} valid)')
print(f'FTIR EC column: {ftir_col} ({addis_filters[ftir_col].notna().sum() if ftir_col else 0} valid)')
print(f'Iron column: {iron_col} ({addis_filters[iron_col].notna().sum() if iron_col else 0} valid)')
print(f'Filter dates: {addis_filters["filter_date"].min()} → {addis_filters["filter_date"].max()}')

In [ ]:
# Merge filter data with daily AERONET
filter_aeronet = addis_filters.copy()
filter_aeronet = filter_aeronet.set_index('filter_date')

# Join with daily SDA (coarse/fine mode)
sda_for_merge = sda_daily[[coarse_col, fine_col, fmf_col]].copy()
aod_for_merge = aod_daily[['AOD_500nm', 'Precipitable_Water(cm)', '440-870_Angstrom_Exponent']].copy()

filter_aeronet = filter_aeronet.join(sda_for_merge, how='left')
filter_aeronet = filter_aeronet.join(aod_for_merge, how='left')

n_matched = filter_aeronet[coarse_col].notna().sum()
print(f'Filter samples matched to AERONET: {n_matched} / {len(filter_aeronet)}')

if hips_col and ftir_col and n_matched > 5:
    # Compute HIPS-FTIR discrepancy
    valid = filter_aeronet.dropna(subset=[hips_col, ftir_col, coarse_col]).copy()
    valid['hips_ec'] = valid[hips_col] / 10  # Fabs / MAC → EC equivalent
    valid['hips_ftir_ratio'] = valid['hips_ec'] / valid[ftir_col]
    valid['hips_ftir_diff'] = valid['hips_ec'] - valid[ftir_col]
    
    print(f'\nSamples with HIPS, FTIR EC, and coarse AOD: {len(valid)}')
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # HIPS/FTIR ratio vs coarse AOD
    ax = axes[0]
    ax.scatter(valid[coarse_col], valid['hips_ftir_ratio'], alpha=0.6, s=40,
              color='#8B4513', edgecolors='k', linewidth=0.3)
    if len(valid) > 5:
        r, p = stats.pearsonr(valid[coarse_col], valid['hips_ftir_ratio'])
        ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}', transform=ax.transAxes,
                fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.axhline(y=1, color='k', linestyle='--', alpha=0.5)
    ax.set_xlabel('Coarse Mode AOD 500nm', fontsize=11)
    ax.set_ylabel('HIPS EC / FTIR EC ratio', fontsize=11)
    ax.set_title('HIPS-FTIR Discrepancy vs\nColumnar Dust Loading', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # HIPS/FTIR ratio vs FMF
    ax = axes[1]
    valid_fmf = valid.dropna(subset=[fmf_col])
    if len(valid_fmf) > 3:
        ax.scatter(valid_fmf[fmf_col], valid_fmf['hips_ftir_ratio'], alpha=0.6, s=40,
                  color='#2196F3', edgecolors='k', linewidth=0.3)
        if len(valid_fmf) > 5:
            r, p = stats.pearsonr(valid_fmf[fmf_col], valid_fmf['hips_ftir_ratio'])
            ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}', transform=ax.transAxes,
                    fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.axhline(y=1, color='k', linestyle='--', alpha=0.5)
    ax.set_xlabel('Fine Mode Fraction (500nm)', fontsize=11)
    ax.set_ylabel('HIPS EC / FTIR EC ratio', fontsize=11)
    ax.set_title('HIPS-FTIR Discrepancy vs\nFine Mode Fraction', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Compare iron (surface) vs coarse AOD (column) as dust proxies
    ax = axes[2]
    if iron_col:
        valid_iron = valid.dropna(subset=[iron_col])
        if len(valid_iron) > 3:
            ax.scatter(valid_iron[coarse_col], valid_iron[iron_col], alpha=0.6, s=40,
                      color='#E67E22', edgecolors='k', linewidth=0.3)
            if len(valid_iron) > 5:
                r, p = stats.pearsonr(valid_iron[coarse_col], valid_iron[iron_col])
                ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}', transform=ax.transAxes,
                        fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
            ax.set_xlabel('Coarse Mode AOD 500nm', fontsize=11)
            ax.set_ylabel('XRF Iron (µg/m³)', fontsize=11)
            ax.set_title('Columnar Dust vs Surface Iron\n(do they agree?)', fontsize=12, fontweight='bold')
            ax.grid(True, alpha=0.3)
    
    plt.suptitle('Dust Interference Diagnostics', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Insufficient matched data for HIPS-FTIR vs AERONET comparison')

---

## Experiment 4: Surface AAE vs Columnar AE — Source Type Consistency

**Surface AAE** (from aethalometer UV/IR ratio) reflects absorption-based source typing:
- AAE ~ 1 → fossil fuel (traffic)
- AAE > 1.5 → biomass burning / brown carbon

**Columnar AE** (from AERONET 440-870nm) reflects extinction-based particle size:
- AE > 1.5 → fine mode dominated (combustion)
- AE < 1 → coarse mode (dust)

These measure different things, but should broadly agree on whether the aerosol regime is combustion- vs dust-dominated. Systematic disagreement tells us the surface and column have different aerosol populations.

In [ ]:
# Compute sub-daily AAE from matched data
valid_ae = matched.dropna(subset=['UV_BCc', 'IR_BCc', '440-870_Angstrom_Exponent']).copy()
valid_ae = valid_ae[(valid_ae['UV_BCc'] > 0) & (valid_ae['IR_BCc'] > 0)]

wavelength_ratio = np.log(880 / 375)
bc_ratio = valid_ae['IR_BCc'] / valid_ae['UV_BCc']
valid_ae['AAE'] = np.log(bc_ratio) / wavelength_ratio
valid_ae['AE'] = valid_ae['440-870_Angstrom_Exponent']

# Filter unreasonable values
valid_ae = valid_ae[(valid_ae['AAE'] > -0.5) & (valid_ae['AAE'] < 3)]

print(f'Matched AAE vs AE: {len(valid_ae)} observations')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter: AAE vs AE
ax = axes[0]
for season in SEASON_ORDER:
    s = valid_ae[valid_ae['season'] == season]
    ax.scatter(s['AE'], s['AAE'], color=SEASON_COLORS[season], alpha=0.3,
              s=15, label=f'{season} (n={len(s)})')
ax.axhline(y=1, color='gray', linestyle=':', alpha=0.5, label='AAE=1 (fossil fuel)')
ax.axvline(x=1, color='gray', linestyle='--', alpha=0.5, label='AE=1 (fine/coarse boundary)')
r, p = stats.pearsonr(valid_ae['AE'], valid_ae['AAE'])
ax.text(0.05, 0.95, f'r={r:.3f}, p={p:.2e}', transform=ax.transAxes,
        fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax.set_xlabel('Columnar AE (440-870nm)', fontsize=11)
ax.set_ylabel('Surface AAE (375-880nm)', fontsize=11)
ax.set_title('Surface vs Columnar Angstrom', fontsize=12, fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Diurnal AAE vs AE
ax = axes[1]
hourly_aae = valid_ae.groupby('hour_local')['AAE'].mean()
hourly_ae = valid_ae.groupby('hour_local')['AE'].mean()
ax.plot(hourly_aae.index, hourly_aae.values, 'o-', color='#E74C3C', linewidth=2, label='Surface AAE')
ax2 = ax.twinx()
ax2.plot(hourly_ae.index, hourly_ae.values, 's-', color='#3498DB', linewidth=2, label='Columnar AE')
ax.set_xlabel('Hour (local)', fontsize=11)
ax.set_ylabel('Surface AAE', fontsize=11, color='#E74C3C')
ax2.set_ylabel('Columnar AE', fontsize=11, color='#3498DB')
ax.set_title('Diurnal AAE vs AE', fontsize=12, fontweight='bold')
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=9)
ax.grid(True, alpha=0.3)

# AE-AAE difference by season
ax = axes[2]
valid_ae['AE_AAE_diff'] = valid_ae['AE'] - valid_ae['AAE']
plot_data = [valid_ae[valid_ae['season'] == s]['AE_AAE_diff'].dropna() for s in SEASON_ORDER]
bp = ax.boxplot(plot_data, labels=SEASON_ORDER, patch_artist=True, showfliers=False)
for patch, season in zip(bp['boxes'], SEASON_ORDER):
    patch.set_facecolor(SEASON_COLORS[season])
    patch.set_alpha(0.7)
ax.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax.set_ylabel('Columnar AE − Surface AAE', fontsize=11)
ax.set_title('AE−AAE Gap by Season\n(+ve = column sees more fine mode)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Surface vs Columnar Angstrom Exponent Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nAAE vs AE statistics by season:')
for season in SEASON_ORDER:
    s = valid_ae[valid_ae['season'] == season]
    print(f'  {season}: AAE mean={s["AAE"].mean():.3f}, AE mean={s["AE"].mean():.3f}, '
          f'gap={s["AE_AAE_diff"].mean():.3f}, n={len(s)}')

---

## Experiment 5: Fine Mode Fraction × BC Regime Analysis

**Logic**: High FMF + High BC = combustion aerosol top-to-bottom (consistent). High FMF + Low BC = fine aerosol aloft but not at surface (elevated layer). Low FMF + High BC = surface BC trapped below dust layer.

Classifying days into these regimes could reveal what's unusual about Addis.

In [ ]:
# Use daily matched data for regime analysis
bc_daily_df = pd.read_pickle(os.path.join(PROCESSED_SITES_DIR, 'df_Addis_Ababa_9am_resampled.pkl'))
bc_daily_df['datetime_local'] = pd.to_datetime(bc_daily_df['datetime_local'])
bc_daily_df.set_index('datetime_local', inplace=True)
bc_daily_df['IR BCc'] = bc_daily_df['IR BCc'] / 1000  # ng → µg
bc_daily_df.loc[bc_daily_df['IR BCc'] < 0, 'IR BCc'] = np.nan

# Make timezone-naive date index for merging
bc_for_merge = bc_daily_df[['IR BCc']].copy()
bc_for_merge.index = bc_for_merge.index.normalize()
if bc_for_merge.index.tz is not None:
    bc_for_merge.index = bc_for_merge.index.tz_localize(None)

# Merge with daily SDA
regime_df = bc_for_merge.join(sda_daily[[coarse_col, fine_col, fmf_col]], how='inner')
regime_df = regime_df.join(aod_daily[['AOD_500nm', 'Precipitable_Water(cm)',
                                       '440-870_Angstrom_Exponent']], how='left')
regime_df = regime_df.dropna(subset=['IR BCc', fmf_col])
regime_df['month'] = regime_df.index.month
regime_df['season'] = regime_df['month'].map(assign_season)

# Classify regimes using median splits
bc_med = regime_df['IR BCc'].median()
fmf_med = regime_df[fmf_col].median()

def classify_regime(row):
    high_bc = row['IR BCc'] > bc_med
    high_fmf = row[fmf_col] > fmf_med
    if high_bc and high_fmf:
        return 'High BC + High FMF\n(combustion throughout)'
    elif high_bc and not high_fmf:
        return 'High BC + Low FMF\n(BC trapped below dust)'
    elif not high_bc and high_fmf:
        return 'Low BC + High FMF\n(elevated fine layer)'
    else:
        return 'Low BC + Low FMF\n(clean/coarse)'

regime_df['regime'] = regime_df.apply(classify_regime, axis=1)

print(f'Regime analysis: {len(regime_df)} matched days')
print(f'  BC median: {bc_med:.3f} µg/m³')
print(f'  FMF median: {fmf_med:.3f}')
print(f'\nRegime counts:')
print(regime_df['regime'].value_counts())

In [ ]:
# Visualize regimes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter: BC vs FMF colored by season
ax = axes[0]
for season in SEASON_ORDER:
    s = regime_df[regime_df['season'] == season]
    ax.scatter(s['IR BCc'], s[fmf_col], color=SEASON_COLORS[season],
              alpha=0.6, s=40, label=season, edgecolors='k', linewidth=0.3)
ax.axhline(y=fmf_med, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=bc_med, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('BC (µg/m³)', fontsize=11)
ax.set_ylabel('Fine Mode Fraction (500nm)', fontsize=11)
ax.set_title('BC vs FMF by Season\n(quadrants = regimes)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Regime frequency by season
ax = axes[1]
regime_season = pd.crosstab(regime_df['season'], regime_df['regime'], normalize='index') * 100
regime_colors = ['#E74C3C', '#E67E22', '#3498DB', '#95A5A6']
regime_season.loc[SEASON_ORDER].plot(kind='bar', stacked=True, ax=ax, color=regime_colors)
ax.set_ylabel('% of days', fontsize=11)
ax.set_title('Regime Distribution by Season', fontsize=12, fontweight='bold')
ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(SEASON_ORDER, rotation=0)
ax.grid(True, alpha=0.3, axis='y')

# Coarse AOD by regime
ax = axes[2]
regimes = regime_df['regime'].unique()
plot_data = [regime_df[regime_df['regime'] == r][coarse_col].dropna() for r in sorted(regimes)]
labels = [r.replace('\n', ' ') for r in sorted(regimes)]
bp = ax.boxplot(plot_data, labels=range(1, len(labels)+1), patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], regime_colors[:len(regimes)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Coarse Mode AOD 500nm', fontsize=11)
ax.set_title('Coarse AOD by Regime', fontsize=12, fontweight='bold')
# Add text labels below
for i, label in enumerate(labels):
    ax.text(i+1, ax.get_ylim()[0] - 0.005, label, ha='center', fontsize=6, rotation=15)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('FMF × BC Regime Classification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Deep dive: what makes the anomalous regimes different?
print('Regime characteristics:')
print('=' * 80)
summary_cols = ['IR BCc', fmf_col, coarse_col, fine_col, 'AOD_500nm',
                'Precipitable_Water(cm)', '440-870_Angstrom_Exponent']
available_cols = [c for c in summary_cols if c in regime_df.columns]

for regime in sorted(regime_df['regime'].unique()):
    rdata = regime_df[regime_df['regime'] == regime]
    print(f'\n{regime.replace(chr(10), " ")} (n={len(rdata)}):')
    print(f'  Season distribution: {dict(rdata["season"].value_counts())}')
    for col in available_cols:
        vals = rdata[col].dropna()
        if len(vals) > 0:
            print(f'  {col}: mean={vals.mean():.4f}, std={vals.std():.4f}')

---

## Experiment 6: Temporal Variability — Does the Surface Track the Column?

For each day with multiple AERONET observations, compute the **intra-day variability** of both BC and AOD. If the surface BC is highly variable but AOD is stable (or vice versa), that tells us about decoupling.

In [ ]:
# Compute intra-day variability (coefficient of variation) for matched observations
daily_stats = matched.groupby('date').agg(
    bc_mean=('IR_BCc', 'mean'),
    bc_std=('IR_BCc', 'std'),
    bc_range=('IR_BCc', lambda x: x.max() - x.min()),
    aod_mean=('AOD_500nm', 'mean'),
    aod_std=('AOD_500nm', 'std'),
    aod_range=('AOD_500nm', lambda x: x.max() - x.min() if x.notna().sum() > 1 else np.nan),
    n_obs=('IR_BCc', 'count'),
    season=('season', 'first'),
).dropna(subset=['bc_std', 'aod_std'])

# Need at least 3 obs per day for meaningful variability
daily_stats = daily_stats[daily_stats['n_obs'] >= 3]

daily_stats['bc_cv'] = daily_stats['bc_std'] / daily_stats['bc_mean']
daily_stats['aod_cv'] = daily_stats['aod_std'] / daily_stats['aod_mean']

print(f'Days with >= 3 matched observations: {len(daily_stats)}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# CV comparison
ax = axes[0]
for season in SEASON_ORDER:
    s = daily_stats[daily_stats['season'] == season]
    ax.scatter(s['aod_cv'], s['bc_cv'], color=SEASON_COLORS[season],
              alpha=0.5, s=30, label=f'{season} (n={len(s)})', edgecolors='k', linewidth=0.3)
ax.plot([0, daily_stats[['aod_cv', 'bc_cv']].max().max()],
        [0, daily_stats[['aod_cv', 'bc_cv']].max().max()], 'k--', alpha=0.3, label='1:1')
r, p = stats.pearsonr(daily_stats['aod_cv'], daily_stats['bc_cv'])
ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}', transform=ax.transAxes,
        fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax.set_xlabel('AOD intra-day CV', fontsize=11)
ax.set_ylabel('BC intra-day CV', fontsize=11)
ax.set_title('Do surface and column\nvary together within days?', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Range comparison
ax = axes[1]
for season in SEASON_ORDER:
    s = daily_stats[daily_stats['season'] == season]
    ax.scatter(s['aod_range'], s['bc_range'], color=SEASON_COLORS[season],
              alpha=0.5, s=30, label=season, edgecolors='k', linewidth=0.3)
ax.set_xlabel('AOD 500nm daily range', fontsize=11)
ax.set_ylabel('BC daily range (µg/m³)', fontsize=11)
ax.set_title('Intra-day Range:\nSurface vs Column', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# BC CV by season
ax = axes[2]
plot_data_bc = [daily_stats[daily_stats['season'] == s]['bc_cv'].dropna() for s in SEASON_ORDER]
plot_data_aod = [daily_stats[daily_stats['season'] == s]['aod_cv'].dropna() for s in SEASON_ORDER]

x = np.arange(len(SEASON_ORDER))
width = 0.35
bars1 = ax.bar(x - width/2, [d.median() for d in plot_data_bc], width,
               label='BC CV', color='#E74C3C', alpha=0.7)
bars2 = ax.bar(x + width/2, [d.median() for d in plot_data_aod], width,
               label='AOD CV', color='#3498DB', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(SEASON_ORDER)
ax.set_ylabel('Median intra-day CV', fontsize=11)
ax.set_title('Surface vs Column Variability\nby Season', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Intra-day Variability: Surface BC vs Columnar AOD', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nMedian intra-day CV by season:')
for season in SEASON_ORDER:
    s = daily_stats[daily_stats['season'] == season]
    print(f'  {season}: BC CV={s["bc_cv"].median():.3f}, AOD CV={s["aod_cv"].median():.3f}, n={len(s)}')

---

## Summary & Interpretation

Collect key findings from all experiments.

In [ ]:
print('=' * 80)
print('AERONET DIAGNOSTICS SUMMARY')
print('=' * 80)
print(f'\nTotal matched sub-daily observations: {len(matched)}')
print(f'Total matched daily regime days: {len(regime_df)}')
print(f'Days with intra-day variability: {len(daily_stats)}')

print('\n--- Key correlations ---')
# Overall BC vs AOD
vm = matched.dropna(subset=['IR_BCc', 'AOD_500nm'])
if len(vm) > 5:
    r, p = stats.pearsonr(vm['IR_BCc'], vm['AOD_500nm'])
    print(f'  Sub-daily BC vs AOD_500nm: r={r:.3f}, p={p:.2e}, n={len(vm)}')

# BC vs Fine mode AOD
vm_fine = matched.dropna(subset=['IR_BCc', 'Fine_Mode_AOD_500nm[tau_f]'])
if len(vm_fine) > 5:
    r, p = stats.pearsonr(vm_fine['IR_BCc'], vm_fine['Fine_Mode_AOD_500nm[tau_f]'])
    print(f'  Sub-daily BC vs Fine AOD: r={r:.3f}, p={p:.2e}, n={len(vm_fine)}')

# BC vs Coarse mode AOD
vm_coarse = matched.dropna(subset=['IR_BCc', 'Coarse_Mode_AOD_500nm[tau_c]'])
if len(vm_coarse) > 5:
    r, p = stats.pearsonr(vm_coarse['IR_BCc'], vm_coarse['Coarse_Mode_AOD_500nm[tau_c]'])
    print(f'  Sub-daily BC vs Coarse AOD: r={r:.3f}, p={p:.2e}, n={len(vm_coarse)}')

print('\n--- What this means for the Addis anomaly ---')
print('  (Interpret results above in context of HIPS/FTIR discrepancy)')
print('  - Strong BC-AOD coupling → surface represents column well')
print('  - Weak coupling → vertical distribution matters, surface anomaly may be real')
print('  - High coarse AOD days with HIPS discrepancy → dust interference confirmed')
print('  - Diurnal decoupling → boundary layer trapping at high elevation')